# Étape 13 — Couverture × justesse × proximité : l'expérience factorielle

Les étapes 10 à 12 suggèrent que l'utilité d'un graphe pour le KT dépend de sa **couverture** et de la **proximité thématique** de ses flèches, et pas de leur **justesse**.
Mais ces graphes différaient sur plusieurs facteurs à la fois. Ici, on les fait varier **séparément**.

**Univers :** les exercices B qui ont au moins un prérequis expert. Chaque B sélectionné reçoit **autant de flèches entrantes que dans le graphe expert** (nombre de flèches identique d'une condition à l'autre).

| Facteur | Niveaux |
|---|---|
| **Couverture** : part des exercices B qui reçoivent des flèches | 25 %, 50 %, 100 % |
| **Source** des flèches entrantes de B | `expert` : les vrais prérequis de B · `theme_faux` : exercices du **même thème** qui ne sont **ni** des prérequis directs **ni** indirects de B · `hors_theme_faux` : exercices d'**autres thèmes**, ni prérequis directs ni indirects |

La condition `expert` à 100 % est exactement le graphe expert complet. On ajoute `aucun` (pas de graphe) comme référence.

**10 graines**, 2 régimes (1 % et 5 % des élèves), PrereqKT : 10 conditions × 2 × 10 = **200 entraînements** (environ 3 h). La boucle avance **graine par graine** : si vous vous arrêtez en route, les graines terminées sont complètes et analysables.

**Hypothèses principales, déclarées avant l'expérience** (mesure : AUC des premières tentatives ; tests appariés par graine ; correction de Holm sur ces 6 tests) :
- **H1 — la couverture compte :** `expert` 100 % > `expert` 25 % ;
- **H2 — la justesse ne compte pas :** `expert` 100 % ≈ `theme_faux` 100 % (test d'**équivalence** TOST, marge = 20 % du gain du graphe expert sur l'absence de graphe) ;
- **H3 — la proximité compte :** `theme_faux` 100 % > `hors_theme_faux` 100 %.

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule F1 — Construction des 9 graphes factoriels (déterministe pour chaque graine)
COUVERTURES = [0.25, 0.5, 1.0]
SOURCES = ['expert', 'theme_faux', 'hors_theme_faux']
GRAINES = list(range(10))

G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
ancetres = {b: nx.ancestors(G_exp, b) for b in range(N)}
prereq_directs = {b: list(G_exp.predecessors(b)) for b in range(N)}
UNIVERS = sorted(b for b in range(N) if prereq_directs[b])
topic = exos.set_index('exo_id')['topic'].to_dict()
meme_theme = {}
for b in range(N):
    meme_theme[b] = [a for a in range(N) if a != b and isinstance(topic[b], str) and topic[a] == topic[b]]

def graphe_factoriel(couv, source, graine):
    rng = random.Random(90_000 + graine)
    ordre = UNIVERS[:]; rng.shuffle(ordre)                 # sous-ensembles emboîtés : 25 % ⊂ 50 % ⊂ 100 %
    choisis = ordre[:int(round(couv * len(ordre)))]
    E, manquants = [], 0
    for b in sorted(choisis):
        n_b = len(prereq_directs[b])
        if source == 'expert':
            E += [(a, b) for a in prereq_directs[b]]
            continue
        interdits = ancetres[b] | {b}
        if source == 'theme_faux':
            pool = [a for a in meme_theme[b] if a not in interdits]
        else:
            pool = [a for a in range(N) if a not in interdits and (not isinstance(topic[b], str) or topic[a] != topic[b])]
        rng_b = random.Random(f'{graine}-{source}-{b}')        # même tirage pour B à toutes les couvertures
        pris = rng_b.sample(pool, min(n_b, len(pool)))
        manquants += n_b - len(pris)
        E += [(a, b) for a in pris]
    return sorted(set(E)), manquants

# Vérifications : nombre de flèches, part des flèches entre exercices du même thème, couverture des premières tentatives
_, Qt, _, Ft = donnees['test']
premieres = pd.Series(np.concatenate([q[f] for q, f in zip(Qt, Ft)])).value_counts()
poids = premieres.reindex(range(N), fill_value=0).values
def couverture_1res(E):
    a = np.zeros(N, bool)
    for _, b in E:
        a[b] = True
    return 100 * (poids * a).sum() / poids.sum()

lignes = []
for couv in COUVERTURES:
    for source in SOURCES:
        m = []
        for g in GRAINES:
            E, manq = graphe_factoriel(couv, source, g)
            meme = np.mean([isinstance(topic[b], str) and topic[a] == topic[b] for a, b in E]) if E else 0
            juste = np.mean([a in ancetres[b] for a, b in E]) if E else 0
            m.append({'fleches': len(E), 'manquantes': manq, 'meme_theme_%': 100 * meme,
                      'prerequis_vrais_%': 100 * juste, 'couverture_1res_%': couverture_1res(E)})
        lignes.append({'couverture': couv, 'source': source, **pd.DataFrame(m).mean().round(1).to_dict()})
verif = pd.DataFrame(lignes)
verif.to_csv(os.path.join(RESULTATS, 'rq2_factoriel_graphes.csv'), index=False)
print(f'Univers : {len(UNIVERS)} exercices avec au moins un prérequis expert')
print(verif.to_string(index=False))

In [ ]:
# Cellule F2 — 200 entraînements, graine par graine (reprise automatique)
FRACTIONS = [0.05, 0.01]
chemin_f = os.path.join(RESULTATS, 'rq2_factoriel.csv')
res_f = pd.read_csv(chemin_f).to_dict('records') if os.path.exists(chemin_f) else []
fait = {(x['condition'], float(x['fraction']), int(x['graine'])) for x in res_f}
CONDITIONS = [('aucun', None, None)] + [(f'{s}_c{int(c*100)}', c, s) for c in COUVERTURES for s in SOURCES]
t0_ = time.time()
for g_ in GRAINES:
    for frac in FRACTIONS:
        for lib, couv, source in CONDITIONS:
            if (lib, frac, g_) in fait:
                continue
            A = graphe('aucun') if couv is None else vers_matrice(graphe_factoriel(couv, source, g_)[0])
            res = entrainer('PrereqKT', A, fraction=frac, graine=g_)
            res_f.append({'condition': lib, 'couverture': couv if couv is not None else 0.0, 'source': source or 'aucun',
                          'fraction': frac, 'graine': g_, **res})
            pd.DataFrame(res_f).to_csv(chemin_f, index=False)
    print(f'Graine {g_} terminée ({(time.time()-t0_)/60:.0f} min écoulées)')
print('Expérience factorielle terminée.')

In [ ]:
# Cellule F3 — Analyse : effets des facteurs (ANOVA à blocs) et hypothèses principales
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

col = 'AUC_1re_test'
df = pd.DataFrame(res_f)
graines_completes = df.groupby('graine').size()
graines_completes = graines_completes[graines_completes == len(CONDITIONS) * len(FRACTIONS)].index
df = df[df.graine.isin(graines_completes)]
print(f'Graines complètes analysées : {sorted(graines_completes.tolist())}')

def ic_bootstrap(d, n=10_000, graine=0):
    rng = np.random.default_rng(graine); d = np.asarray(d)
    m = rng.choice(d, size=(n, len(d)), replace=True).mean(1)
    return np.percentile(m, [2.5, 97.5])

synthese, anova_lignes, hyp = [], [], []
for frac in FRACTIONS:
    s = df[df.fraction == frac]
    piv = s.pivot_table(index='graine', columns='condition', values=col)
    base, exp = piv['aucun'], piv['expert_c100']
    gain = exp - base
    # Tableau des 9 conditions : gain d'AUC sur l'absence de graphe et bénéfice conservé
    for c in COUVERTURES:
        for so in SOURCES:
            lib = f'{so}_c{int(c*100)}'
            d = piv[lib] - base
            lo, hi = ic_bootstrap(d)
            synthese.append({'fraction': frac, 'couverture': c, 'source': so, 'gain_AUC_vs_aucun': round(d.mean(), 4),
                             'IC95_bas': round(lo, 4), 'IC95_haut': round(hi, 4),
                             'benefice_conserve_%': round(100 * d.mean() / gain.mean(), 1)})
    # ANOVA à deux facteurs avec la graine comme bloc (sur le gain par rapport à « aucun »)
    long = s[s.condition != 'aucun'].copy()
    long['gain'] = long[col].values - long['graine'].map(base).values
    mod = smf.ols('gain ~ C(couverture) * C(source) + C(graine)', data=long).fit()
    a = anova_lm(mod, typ=2)
    ss_res = a.loc['Residual', 'sum_sq']
    for terme in ['C(couverture)', 'C(source)', 'C(couverture):C(source)']:
        anova_lignes.append({'fraction': frac, 'facteur': terme.replace('C(', '').replace(')', ''),
                             'F': round(a.loc[terme, 'F'], 2), 'p': a.loc[terme, 'PR(>F)'],
                             'eta2_partiel': round(a.loc[terme, 'sum_sq'] / (a.loc[terme, 'sum_sq'] + ss_res), 3)})
    # Hypothèses principales
    d1 = piv['expert_c100'] - piv['expert_c25']
    hyp.append({'fraction': frac, 'hypothese': 'H1 couverture : expert 100 % > expert 25 %', 'ecart_moyen': d1.mean(),
                'gagne': f'{int((d1 > 0).sum())}/{len(d1)}', 'p': stats.ttest_1samp(d1, 0, alternative='greater').pvalue})
    d2 = piv['expert_c100'] - piv['theme_faux_c100']
    marge = 0.2 * abs(gain.mean())
    p_tost = max(stats.ttest_1samp(d2, -marge, alternative='greater').pvalue,
                 stats.ttest_1samp(d2, marge, alternative='less').pvalue)
    hyp.append({'fraction': frac, 'hypothese': f'H2 justesse : expert 100 % ≈ theme_faux 100 % (TOST, marge ±{marge:.4f})',
                'ecart_moyen': d2.mean(), 'gagne': f'{int((d2 > 0).sum())}/{len(d2)}', 'p': p_tost})
    d3 = piv['theme_faux_c100'] - piv['hors_theme_faux_c100']
    hyp.append({'fraction': frac, 'hypothese': 'H3 proximité : theme_faux 100 % > hors_theme_faux 100 %', 'ecart_moyen': d3.mean(),
                'gagne': f'{int((d3 > 0).sum())}/{len(d3)}', 'p': stats.ttest_1samp(d3, 0, alternative='greater').pvalue})

synthese = pd.DataFrame(synthese); anova_t = pd.DataFrame(anova_lignes); hyp = pd.DataFrame(hyp)
o = hyp['p'].sort_values(); m = len(o)
hyp['p_holm'] = (o * (m - np.arange(m))).cummax().clip(upper=1).reindex(hyp.index)
hyp['confirmee_holm_5%'] = hyp['p_holm'] < 0.05
for t in (anova_t, hyp):
    for c_ in [c for c in t.columns if c.startswith('p')]:
        t[c_] = t[c_].astype(float).round(4)
hyp['ecart_moyen'] = hyp['ecart_moyen'].round(4)
synthese.to_csv(os.path.join(RESULTATS, 'rq2_factoriel_synthese.csv'), index=False)
anova_t.to_csv(os.path.join(RESULTATS, 'rq2_factoriel_anova.csv'), index=False)
hyp.to_csv(os.path.join(RESULTATS, 'rq2_factoriel_hypotheses.csv'), index=False)

for frac in FRACTIONS:
    print(f'\n===== {frac:.0%} des élèves : gain d\'AUC (1res tentatives) par rapport à « aucun », IC 95 % bootstrap =====')
    print(synthese[synthese.fraction == frac].drop(columns='fraction').to_string(index=False))
print('\n===== ANOVA à deux facteurs (bloc = graine) =====')
print(anova_t.to_string(index=False))
print('\n===== Hypothèses principales (Holm sur 6 tests) =====')
print(hyp.to_string(index=False))

In [ ]:
# Cellule F4 — Figure d'interaction : gain d'AUC selon la couverture, une courbe par source
import matplotlib.pyplot as plt
NOMS = {'expert': 'vrais prérequis (expert)', 'theme_faux': 'faux, même thème', 'hors_theme_faux': 'faux, autre thème'}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), sharey=False)
for ax, frac in zip(axes, FRACTIONS):
    s = synthese[synthese.fraction == frac]
    for so, mk in zip(SOURCES, ['o', 's', '^']):
        t = s[s.source == so].sort_values('couverture')
        x = 100 * t.couverture
        ax.plot(x, t.gain_AUC_vs_aucun, marker=mk, label=NOMS[so])
        ax.fill_between(x, t.IC95_bas, t.IC95_haut, alpha=0.15)
    ax.axhline(0, ls=':', c='black', lw=1)
    ax.set_xticks([25, 50, 100]); ax.set_xlabel('Couverture (% des exercices qui reçoivent des prérequis)')
    ax.set_title(f'{frac:.0%} des élèves')
axes[0].set_ylabel("Gain d'AUC sur les 1res tentatives\n(par rapport à « aucun graphe »)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(RESULTATS, 'rq2_factoriel.png'), dpi=200)
plt.show()

In [ ]:
# Cellule F5 — Analyse complémentaire : comparaisons deux à deux entre sources (sans entraînement)
#   Après une déconnexion : relancez la cellule 1, puis celle-ci.
#   « toutes » = moyenne sur les 3 couvertures (c'est l'effet « source » de l'ANOVA). Holm séparé : 3 tests « toutes », 9 tests par couverture.
df5 = pd.read_csv(os.path.join(RESULTATS, 'rq2_factoriel.csv'))
base5 = df5[df5.condition == 'aucun'][['fraction', 'graine', 'AUC_1re_test']].rename(columns={'AUC_1re_test': 'base'})
df5 = df5[df5.condition != 'aucun'].merge(base5, on=['fraction', 'graine'])
df5['gain'] = df5['AUC_1re_test'] - df5['base']

def holm(p):
    p = pd.Series(p); o = p.sort_values(); m = len(o)
    return (o * (m - np.arange(m))).cummax().clip(upper=1).reindex(p.index).values

rng5 = np.random.default_rng(0)
lignes = []
for frac in [0.05, 0.01]:
    for couv in [0.25, 0.5, 1.0, 'toutes']:
        s = df5[df5.fraction == frac] if couv == 'toutes' else df5[(df5.fraction == frac) & (df5.couverture == couv)]
        p = s.groupby(['graine', 'source'])['gain'].mean().unstack()
        for a, b in [('expert', 'theme_faux'), ('expert', 'hors_theme_faux'), ('theme_faux', 'hors_theme_faux')]:
            x = (p[a] - p[b]).values
            bs = rng5.choice(x, size=(5000, len(x)), replace=True).mean(1)
            lignes.append({'fraction': frac, 'couverture': couv, 'comparaison': f'{a} - {b}', 'ecart': x.mean(),
                           'IC95_bas': np.percentile(bs, 2.5), 'IC95_haut': np.percentile(bs, 97.5),
                           'graines_pour': int((x > 0).sum()), 'd_z': x.mean() / x.std(ddof=1),
                           'p_t': stats.ttest_rel(p[a], p[b]).pvalue, 'p_wilcoxon': stats.wilcoxon(p[a], p[b]).pvalue})
paires = pd.DataFrame(lignes)
for frac in [0.05, 0.01]:
    for groupe in [paires.couverture == 'toutes', paires.couverture != 'toutes']:
        m_ = (paires.fraction == frac) & groupe
        paires.loc[m_, 'p_holm'] = holm(paires.loc[m_, 'p_t'].values)
paires = paires.round(4)
paires.to_csv(os.path.join(RESULTATS, 'rq2_factoriel_paires.csv'), index=False)
print(paires.to_string(index=False))